# 🏥 CuraVeris (MedBill AI) — Method 2: LayoutLMv3 Fine-Tuning on Google Colab

This notebook fine-tunes **Microsoft LayoutLMv3** on scanned Indian hospital bills (Apollo, Fortis, Max, etc.) to extract 2D spatial tabular entities:
- `B-ITEM` / `I-ITEM`: Medical description / Surgery / Medicine
- `B-QTY`: Quantity
- `B-RATE`: Unit price in INR (₹)
- `B-AMOUNT`: Line item total in INR (₹)
- `B-DATE`: Date of service
- `B-TOTAL`: Grand invoice total in INR (₹)

**Hardware**: Set Colab runtime to **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# 1. Verify GPU acceleration
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Switch to T4 GPU in Runtime settings for 10x faster training!")

In [ ]:
# 2. Install dependencies
!pip install -q transformers datasets seqeval evaluate pytesseract pillow accelerate torchvision

In [ ]:
# 3. Define Token Labels & Entity Taxonomy
LABELS = [
    "O",
    "B-ITEM", "I-ITEM",
    "B-QTY", "I-QTY",
    "B-RATE", "I-RATE",
    "B-AMOUNT", "I-AMOUNT",
    "B-DATE", "I-DATE",
    "B-DOCTOR", "I-DOCTOR",
    "B-TOTAL", "I-TOTAL"
]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for i, l in enumerate(LABELS)}
print(f"Total entity classes: {len(LABELS)}")

In [ ]:
# 4. Load LayoutLMv3 Base Model & Processor
from transformers import LayoutLMv3ForTokenClassification, LayoutLMv3Processor

processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)
model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)
print("Loaded microsoft/layoutlmv3-base foundation model successfully!")

In [ ]:
# 5. Setup Evaluation Metrics (Seqeval Entity-level Precision / Recall / F1)
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [LABELS[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [LABELS[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }
print("Seqeval metric ready.")

In [ ]:
# 6. Training Configuration
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./curaveris-layoutlmv3-checkpoints",
    num_train_epochs=20,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,               # Standard fine-tuning rate for LayoutLMv3
    warmup_ratio=0.10,                # 10% warmup steps prevent gradient explosion
    weight_decay=0.01,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    fp16=torch.cuda.is_available()
)
print("Training configuration initialized.")

### 7. Export Trained Weights for CuraVeris Backend
Once training completes, run the cell below to compress and download the weights. Then place them in your `backend/app/ml/weights/` folder!

In [ ]:
# Save final model
model.save_pretrained("./final_layoutlmv3_curaveris")
processor.save_pretrained("./final_layoutlmv3_curaveris")

# Zip for easy download
!zip -r curaveris_layoutlmv3_weights.zip ./final_layoutlmv3_curaveris
from google.colab import files
files.download("curaveris_layoutlmv3_weights.zip")
print("Download initiated! Place extracted files in backend/app/ml/weights/")